# 🐼 Pandas Data Analysis: Transaktionsdaten aggregieren, filtern und gruppieren

**Lernziel:** Sie nutzen Pandas für die Analyse von Transaktionsdaten:
- Laden von CSV/JSON in DataFrames
- Filtern, Gruppieren und Aggregieren
- Zeitreihenanalyse mit Datumsindizes
- Erstellung von Compliance‑Berichten (z. B. verdächtige Transaktionen)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Synthetische Transaktionsdaten erzeugen
np.random.seed(42)
n = 500
dates = pd.date_range('2026-01-01', periods=n, freq='D')
amounts = np.random.gamma(2, 500, n)
amounts = np.round(amounts, 2)
countries = np.random.choice(['DE', 'FR', 'IT', 'ES', 'KY', 'CH'], n, p=[0.6, 0.1, 0.1, 0.1, 0.05, 0.05])
status = np.random.choice(['Completed', 'Pending', 'Rejected'], n, p=[0.8, 0.15, 0.05])

df = pd.DataFrame({
    'date': dates,
    'amount': amounts,
    'country': countries,
    'status': status
})

print("Erste 5 Zeilen:")
print(df.head())
print(f"\nGesamtvolumen: {df['amount'].sum():,.2f} EUR")
print(f"Durchschnittlicher Betrag: {df['amount'].mean():.2f} EUR")

## 1. Filtern nach Bedingungen (z. B. hohe Beträge, riskante Länder)

In [ ]:
# Transaktionen über 10.000 EUR
high_value = df[df['amount'] > 10000]
print(f"Anzahl High‑Value Transaktionen: {len(high_value)}")
print(high_value[['date', 'amount', 'country']].head())

# Transaktionen aus Hochrisiko‑Ländern (KY = Cayman Islands)
high_risk_countries = df[df['country'].isin(['KY'])]
print(f"\nTransaktionen aus Hochrisiko‑Ländern: {len(high_risk_countries)}")
print(high_risk_countries[['date', 'amount', 'country']].head())

## 2. Gruppieren und Aggregieren (z. B. pro Land, pro Status)

In [ ]:
# Gesamtvolumen pro Land
volume_by_country = df.groupby('country')['amount'].sum().sort_values(ascending=False)
print("Volumen pro Land:")
print(volume_by_country)

# Mittlerer Betrag pro Status
avg_by_status = df.groupby('status')['amount'].mean()
print("\nMittlerer Betrag pro Status:")
print(avg_by_status)

## 3. Mehrfachaggregationen (z. B. Anzahl, Summe, Mittelwert pro Land)

In [ ]:
summary = df.groupby('country').agg(
    count=('amount', 'count'),
    total_volume=('amount', 'sum'),
    avg_amount=('amount', 'mean')
).round(2)
print(summary)

## 4. Zeitreihenanalyse: Monatliche Entwicklung

In [ ]:
# Datum als Index setzen
df.set_index('date', inplace=True)

# Monatlich aggregieren
monthly = df.resample('M').agg(
    total_volume=('amount', 'sum'),
    count=('amount', 'count')
)

print("Monatliche Aggregation:")
print(monthly.head())

# Plot des monatlichen Volumens
plt.figure(figsize=(10, 4))
plt.plot(monthly.index, monthly['total_volume'], marker='o')
plt.title('Monatliches Transaktionsvolumen')
plt.xlabel('Datum')
plt.ylabel('Volumen in EUR')
plt.grid(True)
plt.show()

## 5. Erstellung eines Compliance‑Berichts (verdächtige Transaktionen)

In [ ]:
# Verdächtige Transaktionen: Betrag > 10.000 UND Land = 'KY'
suspicious = df[(df['amount'] > 10000) & (df['country'] == 'KY')]
print(f"Anzahl verdächtiger Transaktionen: {len(suspicious)}")
print(suspicious[['amount', 'country']].head())

# Export als CSV (für Audit‑Bericht)
suspicious.to_csv('aml_suspicious_report.csv')
print("\nBericht als 'aml_suspicious_report.csv' gespeichert.")

## Diskussion

- Pandas ermöglicht effiziente, lesbare Datenanalyse ohne komplexe Schleifen.
- Die Kombination aus Filtern, Gruppierungen und Resampling ist ideal für AML‑Compliance‑Berichte.
- In der Praxis würden Sie echte Bank‑Transaktionsdaten (z. B. aus einer CSV) laden – die Methoden bleiben identisch.